# 🔄 ETL Raw → Bronze (Spark Job)
## Crime Data Pipeline

Job de ingestão para execução via Airflow/Spark.

**Objetivo**: Extrair dados brutos e persistir na camada Bronze com metadados de ingestão.

In [ ]:
# Configurações do Spark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# Iniciar sessão Spark
spark = SparkSession.builder \
    .appName("RawToBronze_CrimeData") \
    .getOrCreate()

print(f"Spark Session iniciada: {spark.version}")

In [ ]:
# Configuração de caminhos (detectar raiz do projeto)
import os
from pathlib import Path

def find_project_root() -> Path:
    """Encontra a raiz do projeto SBD2"""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for root in candidates:
        if (root / 'Crime_Data_from_2020_to_Present.csv').exists():
            return root
        if (root / 'data').exists() and (root / 'notebooks').exists():
            return root
    return cwd

PROJECT_ROOT = find_project_root()
RAW_PATH = str(PROJECT_ROOT / "Crime_Data_from_2020_to_Present.csv")
BRONZE_PATH = str(PROJECT_ROOT / "data" / "bronze")
BATCH_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
SOURCE_SYSTEM = "LAPD_Crime_Data"

# Garantir que diretório existe
os.makedirs(BRONZE_PATH, exist_ok=True)

print(f"Projeto: {PROJECT_ROOT}")
print(f"Raw: {RAW_PATH}")
print(f"Bronze: {BRONZE_PATH}")
print(f"Batch: {BATCH_ID}")

In [ ]:
# Carregar dados brutos
df_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(RAW_PATH)

print(f"Dados carregados: {df_raw.count():,} registros")
print(f"Colunas: {len(df_raw.columns)}")

In [ ]:
# Validar estrutura
print("Schema dos dados:")
df_raw.printSchema()

In [ ]:
# Adicionar metadados de ingestão
df_bronze = df_raw \
    .withColumn("_ingestion_timestamp", F.current_timestamp()) \
    .withColumn("_batch_id", F.lit(BATCH_ID)) \
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM)) \
    .withColumn("_source_file", F.lit(RAW_PATH.split("/")[-1])) \
    .withColumn("_row_hash", F.hash(*df_raw.columns))

print("Metadados de ingestão adicionados")

In [ ]:
# Verificar duplicatas
total_records = df_bronze.count()
unique_ids = df_bronze.select("DR_NO").distinct().count()
duplicates = total_records - unique_ids

print(f"Total de registros: {total_records:,}")
print(f"IDs únicos: {unique_ids:,}")
print(f"Duplicatas: {duplicates:,}")

In [ ]:
# Salvar na camada Bronze
df_bronze.write \
    .mode("overwrite") \
    .parquet(f"{BRONZE_PATH}/crime_data_bronze.parquet")

print(f"Dados salvos na camada Bronze!")
print(f"Caminho: {BRONZE_PATH}/crime_data_bronze.parquet")

In [ ]:
# Verificar integridade
df_verify = spark.read.parquet(f"{BRONZE_PATH}/crime_data_bronze.parquet")
verify_count = df_verify.count()

print(f"   Verificação de integridade:")
print(f"   Registros gravados: {total_records:,}")
print(f"   Registros lidos: {verify_count:,}")
print(f"   Status: {'OK' if total_records == verify_count else 'FALHA'}")

In [ ]:
# Finalizar
spark.stop()
print("\n" + "="*50)
print("Job Raw → Bronze concluído com sucesso!")
print("="*50)